# Lab 01: Data Understanding & Data Preparation with Pandas
### CRISP-DM: DU + DP

This lab accompanies the DU + DP portion of CRISP-DM. Do note this is part one or a two part series. Next week we will continue on more Data Processing.

**Assumed from lecture:**
*   numerical vs categorical (nominal/ordinal)
*   independent vs dependent variables
*   why models require numerical inputs.

Your job here is to **apply** those ideas to a messy dataset, make decisions, and justify trade-offs.

## Learning outcomes
By the end of this lab, you should be able to:
1. Quickly inspect a new dataset and identify potential data quality risks.
2. Detect, locate, and quantify missing values (NaN) and explain why they matter.
3. Apply common preparation strategies (drop, fill, interpolate) and justify trade-offs.
4. Filter and subset data correctly using boolean masks (without row-wise loops).
5. Use indexing (`loc`, `iloc`) safely and predictably for preparation work.

## How to use this notebook
- **Do not scroll-and-run.** Attempt the exercise cells first.
- When asked for a decision, write a short rationale in plain English.
- If your answer differs from the provided solution, that’s fine **if your justification is sound**.


## Dataset for this lab
We will use a small **semiconductor wafer** dataset to simulate typical quality issues:
- Missing values (NaN)
- Empty strings / inconsistent entries
- The need to filter, transform, and export a cleaned table

In real projects, the dataset would be larger—your workflow should still look similar.


In [ ]:
import pandas as pd
import numpy as np

data = {
    "wafer_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    "lot_number": ["L1", "L1", "L2", "", "L2", "L3", "L3", "L3", "L4", "L4", "L5", "L5"],
    "defect_count": [5, 2, 2, 0, np.nan, 3, 2, np.nan, 1, 2, 3, np.nan],  # rows 4, 7, 11
    "yield_percent": [92.5, 90.2, 95.0, 93.1, 94.8, 96.0, np.nan, 89.5, 90.0, 88.0, 93.1, 92.0],  # row 6
    "inspection_stage": ["final", "final", "initial", "initial", "final", "final", "initial", "final", "initial", "", "final", "final"]  # row 9 has empty string
}

df = pd.DataFrame(data)
df

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
0,101,L1,5.0,92.5,final
1,102,L1,2.0,90.2,final
2,103,L2,2.0,95.0,initial
3,104,,0.0,93.1,initial
4,105,L2,NaN,94.8,final
5,106,L3,3.0,96.0,final
6,107,L3,2.0,NaN,initial
7,108,L3,NaN,89.5,final
8,109,L4,1.0,90.0,initial
9,110,L4,2.0,88.0,


## CRISP-DM Phase: Data Understanding
Start by answering: **What is in this data, and can we trust it enough to proceed?**

You are not cleaning anything yet. First, you inspect.


In [ ]:
df.head()

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
0,101,L1,5.0,92.5,final
1,102,L1,2.0,90.2,final
2,103,L2,2.0,95.0,initial
3,104,,0.0,93.1,initial
4,105,L2,NaN,94.8,final


In [ ]:
df.shape

(12, 5)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   wafer_id          12 non-null     int64  
 1   lot_number        12 non-null     object 
 2   defect_count      9 non-null      float64
 3   yield_percent     11 non-null     float64
 4   inspection_stage  12 non-null     object 
dtypes: float64(2), int64(1), object(2)
memory usage: 612.0+ bytes


In [ ]:
df.describe(include='all')

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
count,12.000000,12,9.000000,11.000000,12
unique,NaN,6,NaN,NaN,3
top,NaN,L3,NaN,NaN,final
freq,NaN,3,NaN,NaN,7
mean,106.500000,NaN,2.222222,92.200000,NaN
std,3.605551,NaN,1.394433,2.540866,NaN
min,101.000000,NaN,0.000000,88.000000,NaN
25%,103.750000,NaN,2.000000,90.100000,NaN
50%,106.500000,NaN,2.000000,92.500000,NaN
75%,109.250000,NaN,3.000000,93.950000,NaN


In [ ]:
# Use these quick probes to support your reasoning below.
pd.DataFrame({
    'nunique': df.nunique(dropna=False),
    'missing': df.isna().sum(),
}).sort_values('nunique', ascending=False)

,nunique,missing
wafer_id,12,0
yield_percent,11,1
lot_number,6,0
defect_count,6,3
inspection_stage,3,0


### Part DU-1
##### Column semantics and variable roles

In real projects, the hardest part of DU is not running `info()`. It is deciding what each column **means** and how it should be treated.

Use the inspection outputs above to classify each column into **one** of the following roles:

- **Candidate feature (X)**: plausible independent variable
- **Target (y)**: dependent variable you want to predict/explain (if applicable)
- **Identifier**: should generally *not* be used as a feature (IDs, keys)
- **Metadata / operational context**: might be useful, but can also leak information depending on the task
- **Ambiguous**: you are unsure; state what you would need to confirm

Then classify each column’s **data type**:

- Numerical (continuous / discrete)
- Categorical (nominal / ordinal)
- Boolean

In [ ]:
# Exercise DU-1: Variable typing and roles
# Fill in the table below (edit the strings).

du1 = pd.DataFrame({
    'column': df.columns,
    'role': ['' for _ in df.columns],  # e.g., 'feature', 'target', 'identifier', 'metadata', 'ambiguous'
    'type': ['' for _ in df.columns],  # e.g., 'num-cont', 'num-disc', 'cat-nom', 'cat-ord', 'bool'
})
du1

,column,role,type
0,wafer_id,,
1,lot_number,,
2,defect_count,,
3,yield_percent,,
4,inspection_stage,,


In [ ]:
du1 = pd.DataFrame({
    'column': [
        'wafer_id',
        'lot_number',
        'defect_count',
        'yield_percent',
        'inspection_stage'
    ],
    'role': [
        'identifier',
        'metadata',
        'feature',
        'target',
        'feature'
    ],
    'type': [
        'num-disc',
        'cat-nom',
        'num-disc',
        'num-cont',
        'cat-nom'
    ]
})
du1


,column,role,type
0,wafer_id,identifier,num-disc
1,lot_number,metadata,cat-nom
2,defect_count,feature,num-disc
3,yield_percent,target,num-cont
4,inspection_stage,feature,cat-nom


#### Exercise DU-2
##### Feature leakage and 'dangerous columns'

Even before modeling, you should be able to spot columns that are risky as features.

1. Which column(s) could act as **identifiers** (unique or near-unique per row)?
2. Which column(s) might create **leakage** depending on the prediction task (e.g., post-outcome measurements)?
3. Which categorical column(s) have **high cardinality** (many unique values) and may cause one-hot explosion?

1. Which column(s) could act as identifiers (unique or near-unique per row)?
*   Column: wafer_id
*   Reasoning: wafer_id is unique (or near-unique) for each row. Although it is numeric, it does not represent a measurable quantity They are also categorical in intent. However it is critical that they must not be treated as features.

2. Which column(s) might create leakage depending on the prediction task?
*   inspection_stage and potentially defect_count

*   Why inspection_stage is risky:
    *   If the prediction task is to estimate yield before final inspection, then knowing whether a wafer is at the final stage leaks future process information.
    *   This would allow the model to indirectly “see the answer” via process timing.

*   Why defect_count can cause leakage:
    *   If defects are measured after yield is assessed (or jointly determined), then using defect_count to predict yield_percent would leak post-outcome information.
    *   The ambiguity around missing values (NaN vs 0) further increases leakage risk.

3. Which categorical column(s) have high cardinality and may cause one-hot explosion?
*   lot_number (potential risk, context-dependent)
    *   In this small sample, cardinality appears moderate (L1–L5), but in real manufacturing datasets, lot numbers often scale to hundreds or thousands.
    *   One-hot encoding such a variable would dramatically increase feature dimensionality.

*   inspection_stage has very low cardinality (initial, final).
    *   One-hot encoding is safe and unlikely to cause feature explosion.

---
---
---

## CRISP-DM Phase: Data Preparation
Data preparation decisions are trade-offs. Each choice has costs.

A common first step is to handle missing values.


### Part 1 — Detecting and locating missing values
In pandas, missing numeric values are often represented as **NaN**.
Your first job is to:
- detect where NaNs exist
- count them
- decide whether they are acceptable


In [ ]:
nan_defects = df[df["defect_count"].isna()].index
print(f"Indexes with missing defect counts: {nan_defects.tolist()}")

Indexes with missing defect counts: [4, 7, 11]


In [ ]:
df[df["defect_count"].isna()].index

Index([4, 7, 11], dtype='int64')

In [ ]:
df["defect_count"].isna()

,defect_count
0,False
1,False
2,False
3,False
4,True
5,False
6,False
7,True
8,False
9,False


In [ ]:
df[df["defect_count"].isna()]

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
4,105,L2,NaN,94.8,final
7,108,L3,NaN,89.5,final
11,112,L5,NaN,92.0,final


In [ ]:
df[df["defect_count"].isna()].index

Index([4, 7, 11], dtype='int64')

In [ ]:
list(df[df["defect_count"].isna()].index)

[4, 7, 11]

In [ ]:
df[df["defect_count"].notna()]

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
0,101,L1,5.0,92.5,final
1,102,L1,2.0,90.2,final
2,103,L2,2.0,95.0,initial
3,104,,0.0,93.1,initial
5,106,L3,3.0,96.0,final
6,107,L3,2.0,NaN,initial
8,109,L4,1.0,90.0,initial
9,110,L4,2.0,88.0,
10,111,L5,3.0,93.1,final


In [ ]:
df[df.isna().any(axis=1)]

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
4,105,L2,NaN,94.8,final
6,107,L3,2.0,NaN,initial
7,108,L3,NaN,89.5,final
11,112,L5,NaN,92.0,final


In [ ]:
df

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
0,101,L1,5.0,92.5,final
1,102,L1,2.0,90.2,final
2,103,L2,2.0,95.0,initial
3,104,,0.0,93.1,initial
4,105,L2,NaN,94.8,final
5,106,L3,3.0,96.0,final
6,107,L3,2.0,NaN,initial
7,108,L3,NaN,89.5,final
8,109,L4,1.0,90.0,initial
9,110,L4,2.0,88.0,


In [ ]:
df["lot_number"] = df["lot_number"].replace("", np.nan)
df["inspection_stage"] = df["inspection_stage"].replace("", np.nan)

In [ ]:
df = df.replace(r'^\s*$', np.nan, regex=True) # It replaces any cell that contains only whitespace (or is empty) with NaN, across the entire DataFrame.

In [ ]:
df.isna().sum()

,0
wafer_id,0
lot_number,1
defect_count,3
yield_percent,1
inspection_stage,1


Note that if missing values are left unhandled:
*   Silent bias through row dropping
    *   Many downstream analyses (correlation, modeling) will automatically drop rows with NaN, changing the effective dataset without explicit awareness.
*   Incorrect averages and rankings
    *   If missing values are implicitly or explicitly treated as zero later, defect rates will be biased downward, leading to incorrect lot- or stage-level comparisons.

### Exercise DP-1: Missing values decision
1. Which column has the most missing values?

2. For the three strategies below to handle the missing values, and state one benefit + one risk for each.
*   Strategy A
    *   Drop rows with missing values (listwise deletion)
*   Strategy B
    *   Impute missing values using contextual statistics (e.g., median by lot or stage)
*   Strategy C
    *   Impute values and explicitly flag augmented rows


**1. Which column has the most missing values?**
*   defect_count has the most missing values (3 rows).
*   yield_percent has fewer missing values (1 row).

**2. Three strategies to handle missing values (with benefits and risks)**
*   Strategy A: Drop rows with missing values (listwise deletion)
    *   Benefit: Preserves semantic correctness of defect_count by not inventing values.
    *   Risk: Reduces sample size and may introduce selection bias if missingness is systematic (e.g., defects missing more often in certain lots or inspection stages).

*   Strategy B: Impute missing values using contextual statistics (e.g., median by lot or stage)
    *   Benefit: Retains more rows and produces more plausible values than global imputation.
    *   Risk: Introduces artificial smoothing and may weaken real relationships (variance shrinkage). Assumes group statistics are reliable.

*   Strategy C: Impute values and explicitly flag augmented rows
    *   Approach:
        *   Create a binary indicator (e.g., defect_count_augmented):
        *   1 if the original value was missing and has been imputed
        *   0 otherwise
        *   Impute missing defect_count values using a defensible strategy (e.g., group median).

    *   Benefit:
        *   Retains data and makes the assumption explicit.
        *   Allows downstream models or analyses to learn whether imputed rows behave differently.
        *   Preserves transparency and auditability.

    *   Risk:
        *   Increases feature count slightly.
        *   Still relies on the imputation assumption being reasonable.
        *   Requires discipline to ensure the flag is not ignored.

Why Strategy C is often preferred in practice. This approach:
*   Avoids silent bias
*   Makes data preparation choices visible
*   Preserves optionality for downstream modeling
*   Aligns with professional data governance and traceability

### Part 2 — Replacing missing values (fill)
Filling missing values introduces an **assumption**.
Common choices include:
- mean/median (numeric)
- a sentinel value (e.g., 0) **only if it makes domain sense**

The three strategies above were example strategies. Do not assume they are always appropriate. We will practice these strategies here.


In [ ]:
data = {
    "wafer_id": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    "lot_number": ["L1", "L1", "L2", "", "L2", "L3", "L3", "L3", "L4", "L4", "L5", "L5"],
    "defect_count": [5, 2, 2, 0, np.nan, 3, 2, np.nan, 1, 2, 3, np.nan],  # rows 4, 7, 11
    "yield_percent": [92.5, 90.2, 95.0, 93.1, 94.8, 96.0, np.nan, 89.5, 90.0, 88.0, 93.1, 92.0],  # row 6
    "inspection_stage": ["final", "final", "initial", "initial", "final", "final", "initial", "final", "initial", "", "final", "final"]  # row 9 has empty string
}

df = pd.DataFrame(data)

### Strategy A

In [ ]:
# If you're focusing on defect_count only:
df_A_defect = df.dropna(subset=["defect_count"]).copy()

print("Original rows:", len(df))
print("After dropna on defect_count:", len(df_A_defect))

Original rows: 12
After dropna on defect_count: 9


In [ ]:
df_A_defect

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
0,101,L1,5.0,92.5,final
1,102,L1,2.0,90.2,final
2,103,L2,2.0,95.0,initial
3,104,,0.0,93.1,initial
5,106,L3,3.0,96.0,final
6,107,L3,2.0,NaN,initial
8,109,L4,1.0,90.0,initial
9,110,L4,2.0,88.0,
10,111,L5,3.0,93.1,final


### Strategy B

In [ ]:
df_B = df.copy()

# Strategy B: Median by (lot_number, inspection_stage), with fallback to global median
group_keys = ["lot_number", "inspection_stage"]

group_median = df_B.groupby(group_keys)["defect_count"].transform("median")
global_median = df_B["defect_count"].median()

df_B["defect_count_imputed"] = (
    df_B["defect_count"]
    .fillna(group_median)
    .fillna(global_median)
)

# (Optional) Replace the original column to keep the table clean
df_B["defect_count"] = df_B["defect_count_imputed"]
df_B = df_B.drop(columns=["defect_count_imputed"])

print("Any remaining NaN in defect_count after imputation:", df_B["defect_count"].isna().any())

df_B

Any remaining NaN in defect_count after imputation: False


,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
0,101,L1,5.0,92.5,final
1,102,L1,2.0,90.2,final
2,103,L2,2.0,95.0,initial
3,104,,0.0,93.1,initial
4,105,L2,2.0,94.8,final
5,106,L3,3.0,96.0,final
6,107,L3,2.0,NaN,initial
7,108,L3,3.0,89.5,final
8,109,L4,1.0,90.0,initial
9,110,L4,2.0,88.0,


In [ ]:
group_median

,defect_count
0,3.5
1,3.5
2,2.0
3,0.0
4,NaN
5,3.0
6,2.0
7,3.0
8,1.0
9,2.0


In [ ]:
global_median

2.0

### Strategy C

In [ ]:
import numpy as np
import pandas as pd

# 1) Create augmentation flag for defect_count (1 if originally missing)
df["defect_count_augmented"] = df["defect_count"].isna().astype(int)

# 2) Choose an imputation strategy:
#    Use group-median by (lot_number, inspection_stage), then fall back to global median.
group_keys = ["lot_number", "inspection_stage"]

group_median = df.groupby(group_keys)["defect_count"].transform("median")
global_median = df["defect_count"].median()

df["defect_count_imputed"] = df["defect_count"].fillna(group_median).fillna(global_median)

# 3) (Optional but good practice) Keep a single cleaned column and drop intermediates
df["defect_count_clean"] = df["defect_count_imputed"]
df = df.drop(columns=["defect_count_imputed"])

# 4) Quick sanity checks
print("Rows augmented:", df["defect_count_augmented"].sum())
print("Any remaining NaN in defect_count_clean:", df["defect_count_clean"].isna().any())

df

Rows augmented: 3
Any remaining NaN in defect_count_clean: False


,wafer_id,lot_number,defect_count,yield_percent,inspection_stage,defect_count_augmented,defect_count_clean
0,101,L1,5.0,92.5,final,0,5.0
1,102,L1,2.0,90.2,final,0,2.0
2,103,L2,2.0,95.0,initial,0,2.0
3,104,,0.0,93.1,initial,0,0.0
4,105,L2,NaN,94.8,final,1,2.0
5,106,L3,3.0,96.0,final,0,3.0
6,107,L3,2.0,NaN,initial,0,2.0
7,108,L3,NaN,89.5,final,1,3.0
8,109,L4,1.0,90.0,initial,0,1.0
9,110,L4,2.0,88.0,,0,2.0


### Exercise DP-2: Choose and justify a fill strategy
Pick **one** fill strategy from above (or propose your own).
1. Why is it reasonable for this dataset?
2. What bias could it introduce?
3. How would you communicate this assumption to a stakeholder?


No "right" answer. However you can refer to DP-1 for the various justifications.

### Part 3 — Indexing safely with `loc` and `iloc`
Indexing mistakes cause silent bugs (wrong rows, wrong columns).

- `loc[row_labels, col_labels]` is **label-based**
- `iloc[row_positions, col_positions]` is **position-based**

In preparation work, prefer explicit, readable indexing.


In [ ]:
df = df.replace("", np.nan)
df["lot_number"] = df["lot_number"].fillna(-1)
df["inspection_stage"] = df["inspection_stage"].fillna("unknown")
df["defect_count"] = df["defect_count"].fillna(df["defect_count"].mean())
df["yield_percent"] = df["yield_percent"].fillna(df["yield_percent"].mean())

In [ ]:
df

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage,defect_count_augmented,defect_count_clean
0,101,L1,5.000000,92.5,final,0,5.0
1,102,L1,2.000000,90.2,final,0,2.0
2,103,L2,2.000000,95.0,initial,0,2.0
3,104,-1,0.000000,93.1,initial,0,0.0
4,105,L2,2.222222,94.8,final,1,2.0
5,106,L3,3.000000,96.0,final,0,3.0
6,107,L3,2.000000,92.2,initial,0,2.0
7,108,L3,2.222222,89.5,final,1,3.0
8,109,L4,1.000000,90.0,initial,0,1.0
9,110,L4,2.000000,88.0,unknown,0,2.0


In [ ]:
df["wafer_id"] == 103

,wafer_id
0,False
1,False
2,True
3,False
4,False
5,False
6,False
7,False
8,False
9,False


In [ ]:
df.loc[df["wafer_id"] == 103, "lot_number"]

,lot_number
2,L2


In [ ]:
df.loc[df["wafer_id"] >= 110, ["wafer_id", "lot_number", "defect_count"]]

,wafer_id,lot_number,defect_count
9,110,L4,2.000000
10,111,L5,3.000000
11,112,L5,2.222222


In [ ]:
df.loc[2, "lot_number"]   # Row with index label 2, column "lot_number"

'L2'

In [ ]:
df.head(1)

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage,defect_count_augmented,defect_count_clean
0,101,L1,5.0,92.5,final,0,5.0


In [ ]:
df.iloc[2, 1]             # 3rd row (index 2), 2nd column (index 1)

'L2'

In [ ]:
df.iloc[df[df["wafer_id"] >= 110].index, [1,2]]             # 3rd row (index 2), 2nd column (index 1)

,lot_number,defect_count
9,L4,2.000000
10,L5,3.000000
11,L5,2.222222


### Exercise DP-3: Indexing confidence check
1. Use `loc` to select wafers with `wafer_id` 101 to 105 (inclusive) and only the columns `wafer_id`, `lot_number`, `defect_count`.
2. Use `iloc` to select the first 3 rows and first 2 columns.

Write the code yourself before running the solution.


In [ ]:
# Your attempt


In [ ]:
# Solution
df.loc[df['wafer_id'].between(101,105), ['wafer_id','lot_number','defect_count']]

,wafer_id,lot_number,defect_count
0,101,L1,5.000000
1,102,L1,2.000000
2,103,L2,2.000000
3,104,-1,0.000000
4,105,L2,2.222222


In [ ]:
# Solution
df.iloc[0:3, 0:2]

,wafer_id,lot_number
0,101,L1
1,102,L1
2,103,L2


### Part 4 — Filtering using boolean masks
Filtering is a preparation decision: you are defining what data is **relevant**.

**Common mistake:** using `and/or` instead of `&/|`.

Rules:
- Wrap each condition in parentheses
- Use `&` for AND, `|` for OR, `~` for NOT


In [ ]:
# Sample semiconductor data
wafer_ids = ['w001', 'w002', 'w003', 'w004', 'w005']
defect_counts = [5, np.nan, 12, 0, 3]

# Create a dictionary
wafer_data = {
    "wafer_id": wafer_ids,
    "defect_count": defect_counts
}

# Create a DataFrame with custom index
wafer_df = pd.DataFrame(wafer_data, index=['s0', 's1', 's2', 's3', 's4'])

print(wafer_df)


   wafer_id  defect_count
s0     w001           5.0
s1     w002           NaN
s2     w003          12.0
s3     w004           0.0
s4     w005           3.0


In [ ]:
wafer_df.loc['s2':'s4']

,wafer_id,defect_count
s2,w003,12.0
s3,w004,0.0
s4,w005,3.0


In [ ]:
wafer_df[wafer_df.defect_count > 5]

,wafer_id,defect_count
s2,w003,12.0


In [ ]:
wafer_df.defect_count > 5

,defect_count
s0,False
s1,False
s2,True
s3,False
s4,False


In [ ]:
wafer_df.loc[:,['wafer_id']]

,wafer_id
s0,w001
s1,w002
s2,w003
s3,w004
s4,w005


In [ ]:
# Dropping of NA columns can be done in this manner.
# Create a DataFrame with custom index
wafer_df = pd.DataFrame(wafer_data, index=['s0', 's1', 's2', 's3', 's4'])
print(wafer_df)

wafer_df_cleaned = wafer_df.dropna()
print(wafer_df_cleaned)

   wafer_id  defect_count
s0     w001           5.0
s1     w002           NaN
s2     w003          12.0
s3     w004           0.0
s4     w005           3.0
   wafer_id  defect_count
s0     w001           5.0
s2     w003          12.0
s3     w004           0.0
s4     w005           3.0


In [ ]:
wafer_df_filtered = wafer_df[~(wafer_df['wafer_id'] == 'w003')]

In [ ]:
wafer_df_filtered

,wafer_id,defect_count
s0,w001,5.0
s1,w002,NaN
s3,w004,0.0
s4,w005,3.0


In [ ]:
wafer_df['wafer_id'] == 'w003'

,wafer_id
s0,False
s1,False
s2,True
s3,False
s4,False


In [ ]:
~(wafer_df['wafer_id'] == 'w003')

,wafer_id
s0,True
s1,True
s2,False
s3,True
s4,True


In [ ]:
wafer_df[~(wafer_df['wafer_id'] == 'w003')]

,wafer_id,defect_count
s0,w001,5.0
s1,w002,NaN
s3,w004,0.0
s4,w005,3.0


In [ ]:
wafer_df.drop(wafer_df[wafer_df['wafer_id'] == 'w003'].index)

,wafer_id,defect_count
s0,w001,5.0
s1,w002,NaN
s3,w004,0.0
s4,w005,3.0


### Exercise DP-4: Filtering with trade-offs
Scenario: you suspect that empty `lot_number` entries are data-entry errors.

1. Filter out rows with empty `lot_number`.
2. How many rows did you remove?

In [ ]:
# Your attempt


In [ ]:
# Solution
df_nonempty_lot = df[df['lot_number'].astype(str).str.len() > 0]
df_nonempty_lot

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
0,101,L1,5.0,92.5,final
1,102,L1,2.0,90.2,final
2,103,L2,2.0,95.0,initial
3,104,NaN,0.0,93.1,initial
4,105,L2,NaN,94.8,final
5,106,L3,3.0,96.0,final
6,107,L3,2.0,NaN,initial
7,108,L3,NaN,89.5,final
8,109,L4,1.0,90.0,initial
9,110,L4,2.0,88.0,NaN


In [ ]:
# Rows removed
len(df) - len(df_nonempty_lot)

0

### Exercise DP-5: Debug this filtering code
This fails:
```python
df[df['defect_count'] > 2 and df['lot_number'] == 'L3']
```
1. Explain why it fails.
2. Fix it.


In [ ]:
# Write your explanation and fix here.
# Explanation:
#
# Fix:
df[(df['defect_count'] > 2) & (df['lot_number'] == 'L3')]

,wafer_id,lot_number,defect_count,yield_percent,inspection_stage
5,106,L3,3.0,96.0,final


As the lab is already quite long, we will leave the next part of Data Preparation to next week....